# Mini projekt: Analiza Palmer Penguins dataseta

## Pitanje koje rješavamo

**Možemo li pomoću mjerenja tijela pingvina razlikovati vrste pingvina i koje mjere najviše pomažu u toj razlici?**

U ovom primjeru koristimo znanje iz:

- **Pandas**: učitavanje podataka, pregled, čišćenje, filtriranje, grupiranje, pivot tablice i `plot()`
- **NumPy**: rad s numeričkim nizovima, izračun udaljenosti, standardizacija podataka i osnovna klasifikacija
- **Matplotlib / Pandas plot**: osnovne vizualizacije podataka

Dataset: Palmer Penguins  
Izvor CSV datoteke: `palmerpenguins` projekt na GitHubu.

## 1. Učitavanje biblioteka i podataka

Dataset sadrži mjerenja za tri vrste pingvina:

- Adelie
- Chinstrap
- Gentoo

Varijable koje ćemo koristiti:

- `species` – vrsta pingvina
- `island` – otok
- `bill_length_mm` – duljina kljuna
- `bill_depth_mm` – dubina kljuna
- `flipper_length_mm` – duljina peraje
- `body_mass_g` – masa tijela
- `sex` – spol
- `year` – godina mjerenja

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"

df = pd.read_csv(url)
df.head()

## 2. Prvi pregled podataka

Prije bilo kakve analize trebamo razumjeti podatke:

- koliko redaka i stupaca imamo
- koji su tipovi podataka
- postoje li nedostajuće vrijednosti
- kako izgledaju osnovne statistike

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

## 3. Čišćenje podataka

Za ovu analizu trebaju nam potpuni podaci za sljedeće stupce:

- `species`
- `bill_length_mm`
- `bill_depth_mm`
- `flipper_length_mm`
- `body_mass_g`
- `sex`

U stvarnim projektima ne brišemo uvijek redove s nedostajućim vrijednostima. Nekada ih nadomještamo prosjekom, medijanom ili posebnim pravilima.  
Ovdje ćemo radi jednostavnosti maknuti redove koji nemaju potrebne podatke.

In [ ]:
required_columns = [
    "species",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "sex"
]

clean_df = df.dropna(subset=required_columns).copy()

print("Originalni broj redaka:", len(df))
print("Broj redaka nakon čišćenja:", len(clean_df))
print("Uklonjeno redaka:", len(df) - len(clean_df))

In [ ]:
clean_df.isna().sum()

## 4. Jednostavna analiza: koliko pingvina imamo po vrsti?

Ovo je osnovno pitanje prije dublje analize. Ako jedna vrsta ima puno više zapisa od druge, to može utjecati na interpretaciju rezultata.

In [ ]:
species_counts = clean_df["species"].value_counts()
species_counts

In [ ]:
species_counts.plot(
    kind="bar",
    title="Broj pingvina po vrsti",
    xlabel="Vrsta",
    ylabel="Broj pingvina"
)

plt.xticks(rotation=0)
plt.show()

## 5. Grupiranje podataka po vrsti

Sada gledamo prosječne vrijednosti mjerenja za svaku vrstu pingvina.

Ovo je tipičan Pandas korak:

1. odaberemo podatke
2. grupiramo ih po nekoj kategoriji
3. računamo agregirane vrijednosti

In [ ]:
numeric_columns = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

species_means = clean_df.groupby("species")[numeric_columns].mean()
species_means

In [ ]:
species_means.plot(
    kind="bar",
    title="Prosječna mjerenja po vrsti pingvina",
    figsize=(10, 5)
)

plt.ylabel("Vrijednost")
plt.xticks(rotation=0)
plt.show()

### Problem s prethodnim grafikonom

Na prethodnom grafikonu uspoređujemo različite mjerne jedinice:

- milimetre
- grame

Zato masa tijela dominira grafikonom.

Rješenje je napraviti **standardizaciju** podataka.

## 6. NumPy: standardizacija podataka

Standardizacija znači da svaku vrijednost pretvaramo u oblik:

```text
standardizirana vrijednost = (vrijednost - prosjek) / standardna devijacija
```

Nakon toga sve varijable imaju usporedivu skalu.

In [ ]:
measurements = clean_df[numeric_columns].to_numpy()

means = np.mean(measurements, axis=0)
stds = np.std(measurements, axis=0)

standardized = (measurements - means) / stds

standardized[:5]

In [ ]:
standardized_df = pd.DataFrame(
    standardized,
    columns=[col + "_std" for col in numeric_columns],
    index=clean_df.index
)

analysis_df = pd.concat([clean_df, standardized_df], axis=1)
analysis_df.head()

Sada možemo ponovno usporediti prosječne vrijednosti po vrsti, ali na standardiziranoj skali.

In [ ]:
standardized_columns = [col + "_std" for col in numeric_columns]

standardized_means = analysis_df.groupby("species")[standardized_columns].mean()
standardized_means

In [ ]:
standardized_means.plot(
    kind="bar",
    title="Standardizirana prosječna mjerenja po vrsti pingvina",
    figsize=(10, 5)
)

plt.axhline(0, linewidth=1)
plt.ylabel("Standardizirana vrijednost")
plt.xticks(rotation=0)
plt.show()

## 7. Vizualna provjera: koje mjere najbolje razdvajaju vrste?

Pokušajmo vidjeti odnos između dvije varijable:

- duljina peraje
- masa tijela

Ako se vrste jasno grupiraju, te varijable mogu biti korisne za razlikovanje vrsta.

In [ ]:
for species in analysis_df["species"].unique():
    subset = analysis_df[analysis_df["species"] == species]
    subset.plot.scatter(
        x="flipper_length_mm",
        y="body_mass_g",
        label=species,
        figsize=(8, 5),
        title="Odnos duljine peraje i mase tijela po vrsti"
    )

plt.xlabel("Duljina peraje (mm)")
plt.ylabel("Masa tijela (g)")
plt.show()

Prethodni pristup otvara više zasebnih grafova jer `DataFrame.plot.scatter()` ne grupira automatski po kategoriji na način kao neke druge biblioteke.

Zato ćemo za jedan zajednički graf koristiti obični `matplotlib`, ali i dalje koristimo Pandas za filtriranje podataka.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for species in analysis_df["species"].unique():
    subset = analysis_df[analysis_df["species"] == species]
    ax.scatter(
        subset["flipper_length_mm"],
        subset["body_mass_g"],
        label=species
    )

ax.set_title("Odnos duljine peraje i mase tijela po vrsti")
ax.set_xlabel("Duljina peraje (mm)")
ax.set_ylabel("Masa tijela (g)")
ax.legend()
plt.show()

## 8. Korelacija numeričkih varijabli

Korelacija nam pokazuje koliko su dvije numeričke varijable povezane.

Vrijednost korelacije može biti:

- blizu `1`: jaka pozitivna povezanost
- blizu `-1`: jaka negativna povezanost
- blizu `0`: slaba linearna povezanost

In [ ]:
correlation_matrix = clean_df[numeric_columns].corr()
correlation_matrix

In [ ]:
correlation_matrix.plot(
    kind="bar",
    figsize=(10, 5),
    title="Korelacije između numeričkih varijabli"
)

plt.xticks(rotation=45)
plt.show()

## 9. Pivot tablica: prosječna masa po vrsti i spolu

Ovo je dobar primjer kako pomoću Pandasa brzo dobiti pregled podataka po dvije kategorije.

In [ ]:
pivot_mass = clean_df.pivot_table(
    values="body_mass_g",
    index="species",
    columns="sex",
    aggfunc="mean"
)

pivot_mass

In [ ]:
pivot_mass.plot(
    kind="bar",
    title="Prosječna masa tijela po vrsti i spolu",
    ylabel="Masa tijela (g)",
    figsize=(8, 5)
)

plt.xticks(rotation=0)
plt.show()

## 10. Mini model s NumPy: najbliži prosjek vrste

Sada ćemo napraviti vrlo jednostavan model.

Ideja:

1. za svaku vrstu izračunamo prosječne standardizirane vrijednosti
2. za svakog pingvina izračunamo udaljenost do prosjeka svake vrste
3. pingvina dodijelimo vrsti čijem je prosjeku najbliži

Ovo nije pravi produkcijski model strojnog učenja, nego jednostavan primjer kako NumPy može pomoći u rješavanju analitičkog problema.

In [ ]:
features = analysis_df[standardized_columns].to_numpy()
species_names = sorted(analysis_df["species"].unique())

centroids = []

for species in species_names:
    species_features = analysis_df.loc[
        analysis_df["species"] == species,
        standardized_columns
    ].to_numpy()

    centroid = np.mean(species_features, axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)

centroids

In [ ]:
# distances ima oblik:
# broj_pingvina x broj_vrsta

distances = np.sqrt(((features[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2))

distances[:5]

In [ ]:
nearest_centroid_index = np.argmin(distances, axis=1)

predicted_species = np.array(species_names)[nearest_centroid_index]

analysis_df["predicted_species"] = predicted_species

analysis_df[["species", "predicted_species"]].head(10)

## 11. Procjena rezultata

Sada ćemo vidjeti koliko često je naš jednostavni model pogodio vrstu pingvina.

In [ ]:
analysis_df["correct_prediction"] = analysis_df["species"] == analysis_df["predicted_species"]

accuracy = analysis_df["correct_prediction"].mean()

print(f"Točnost jednostavnog modela: {accuracy:.2%}")

In [ ]:
confusion_table = pd.crosstab(
    analysis_df["species"],
    analysis_df["predicted_species"],
    rownames=["Stvarna vrsta"],
    colnames=["Predviđena vrsta"]
)

confusion_table

In [ ]:
confusion_table.plot(
    kind="bar",
    figsize=(8, 5),
    title="Stvarne i predviđene vrste pingvina"
)

plt.ylabel("Broj pingvina")
plt.xticks(rotation=0)
plt.show()

## 12. Zaključak

Na temelju analize možemo zaključiti:

1. Dataset je trebalo očistiti jer su neke vrijednosti nedostajale.
2. Vrste pingvina se razlikuju po prosječnoj masi, duljini peraje i dimenzijama kljuna.
3. Duljina peraje i masa tijela vizualno dobro odvajaju Gentoo pingvine od ostalih.
4. Za usporedbu varijabli različitih mjernih jedinica korisno je napraviti standardizaciju pomoću NumPyja.
5. Vrlo jednostavan model "najbližeg prosjeka vrste" može dati solidan prvi rezultat, ali nije zamjena za ozbiljniji model strojnog učenja.

## Pitanje za polaznike

Kako biste poboljšali ovaj model?

Moguće ideje:

- koristiti samo neke varijable
- odvojiti podatke na trening i test skup
- posebno analizirati mužjake i ženke
- usporediti rezultate prije i nakon standardizacije
- dodati dodatne vizualizacije

## Dodatni zadaci za vježbu

1. Prikažite prosječnu duljinu kljuna po vrsti pingvina.
2. Prikažite broj pingvina po otoku.
3. Izračunajte minimalnu, maksimalnu i prosječnu masu po vrsti.
4. Napravite pivot tablicu koja prikazuje prosječnu duljinu peraje po vrsti i spolu.
5. Napravite novi jednostavni model koji koristi samo dvije varijable: `flipper_length_mm` i `body_mass_g`.
6. Usporedite točnost modela sa svim varijablama i modela sa samo dvije varijable.